## Ejercicio 4

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lista4.data import load_csv
from lista4.models import fit_linear
from lista4.metrics import *

pd.set_option("display.float_format", lambda v: f"{v:,.6f}")

In [4]:
df4 = load_csv("clientes.csv")

variables_disponibles = [
    "antiguedad_meses",     # disponible en t0
    "uso_30d_previos",      # datos entre t0-30 y t0
    "facturas_vencidas_180d",  # datos entre t0-180 y t0
    "tickets_90d",          # datos entre t0-90 y t0
    "pago_automatico",      # disponible en t0  (ajusta el nombre real de la columna)
]

# Ajusta los nombres según el CSV real:
print(df4.columns.tolist())

['antiguedad_meses', 'uso_30d_previos', 'facturas_vencidas_180d', 'tickets_90d', 'pago_automatico', 'ajuste_comercial_30d', 'abandona_30d']


In [5]:
from lista4.models import fit_logistic

y_col = "abandona_30d"
todas = [c for c in df4.columns if c != y_col]

M1 = fit_logistic(df4, todas, y_col=y_col)          # todas las predictoras
M2 = fit_logistic(df4, variables_disponibles, y_col=y_col)  # solo disponibles en t0

In [9]:
def evaluar(model, df, x_cols, y_col="abandona_30d"):
    X = df[x_cols].to_numpy(dtype=float)
    y = df[y_col].to_numpy(dtype=float)
    p = model.predict_proba(X)[:, 1]
    yhat = (p >= 0.5).astype(int)
    R = logistic_risk(y, p)
    errs, prop = misclassification(y, yhat)
    return {"R_logistico": R, "n_errores": errs, "prop_errores": prop}

res_M1 = evaluar(M1, df4, todas)
res_M2 = evaluar(M2, df4, variables_disponibles)

In [10]:
tabla_4d = pd.DataFrame([
    {"modelo": "M1 (todas)", **res_M1},
    {"modelo": "M2 (disponibles en t0)", **res_M2},
])
tabla_4d

,modelo,R_logistico,n_errores,prop_errores
0,M1 (todas),0.342485,160,0.133333
1,M2 (disponibles en t0),0.571716,336,0.280000
